# 🐼 pandas Bootcamp — All the Core Concepts (one notebook)

A complete, hands-on tour of **NumPy + pandas** — the toolkit for wrangling
tabular data in Python — designed to run **inside Databricks** (or any Jupyter
environment). Every cell is self-contained: the notebook **builds its own sample
data**, so there's nothing to upload.

**How to use it**
1. Import into Databricks: *Workspace → Import → File* → this `.ipynb`.
2. Attach to any cluster / serverless compute (pandas & NumPy are pre-installed).
3. Run cells **top to bottom** — later sections use the DataFrames created near
   the top. Read, run, tweak, re-run.

**Covered:** NumPy arrays & vectorization, `Series`/`DataFrame`, selecting with
`.loc`/`.iloc`, filtering, deriving columns, sorting, missing data, aggregation
& `groupby`, `apply`/`map`, `merge`/`concat`, `drop_duplicates`, reshaping
(`pivot`/`melt`), the `.str` and `.dt` accessors, **time series**
(`resample`/`rolling`/`shift`), categorical dtype, and IO — plus how pandas
connects to Spark on Databricks.

## 1 · NumPy foundations

pandas is built on **NumPy**. A NumPy `ndarray` stores numbers compactly and runs
operations in fast C loops — **vectorization** (no Python `for` loop). This is
why pandas is fast.

In [ ]:
import numpy as np

prices = np.array([19.99, 5.0, 120.0, 3.5])
print("array:", prices, "| dtype:", prices.dtype)
print("with tax:", np.round(prices * 1.2, 2))          # vectorized, elementwise
print("sum:", prices.sum(), "| mean:", prices.mean(), "| max:", prices.max())

# Boolean masks filter; np.where is a vectorized if/else
amounts = np.array([120, -5, 89, 0, 240, -12])
print("valid:", amounts[amounts > 0])
print("labels:", np.where(amounts >= 100, "big", "small"))

# Broadcasting + aggregation along an axis
m = np.array([[1, 2, 3], [4, 5, 6]])
print("m + 10:\n", m + 10)
print("sum per row:", m.sum(axis=1), "| per col:", m.sum(axis=0))

## 2 · Build the sample data

We create a small fictional retailer as DataFrames — no files needed. `customers`
and `products` are small and explicit; `orders` and `order_items` are generated
so aggregations and time series are meaningful.

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(42)

customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "name": ["Ava Smith","Liam Patel","Mei Kim","Noah Garcia",
             "Olivia Rossi","Raj Haddad","Sofia Silva","Chen Wu"],
    "country": ["US","GB","de","US","IT","in","br","US"],   # messy casing
    "email": ["ava@x.com","liam@x.com", None, "noah@x.com",
              "olivia@x.com", None, "sofia@x.com","chen@x.com"],
    "signup_date": pd.to_datetime(["2023-01-15","2023-03-02","2023-05-20",
        "2023-06-11","2023-08-09","2023-09-30","2024-01-05","2024-02-18"]),
})

products = pd.DataFrame({
    "product_id": [101,102,103,104,105,106,107,108],
    "product_name": ["Wireless Mouse","Mechanical Keyboard","Novel","Coffee Mug",
                     "Desk Lamp","Lego Set","Water Bottle","Headphones"],
    "category": ["Electronics","Electronics","Books","Home",
                 "Home","Toys","Home","Electronics"],
    "unit_price": [24.99, 79.50, 14.00, 9.75, 39.90, 59.99, 12.50, 129.00],
})

n = 200
orders = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "customer_id": rng.integers(1, 9, n),
    "order_ts": pd.Timestamp("2024-01-01") + pd.to_timedelta(rng.integers(0, 365, n), unit="D"),
    "status": rng.choice(["completed","returned","cancelled"], n, p=[0.8, 0.12, 0.08]),
    "amount": np.round(rng.uniform(10, 400, n), 2),
})

item_rows = []
for oid in orders["order_id"]:
    for _ in range(rng.integers(1, 4)):
        item_rows.append((oid, int(rng.integers(101, 109)), int(rng.integers(1, 5))))
order_items = pd.DataFrame(item_rows, columns=["order_id","product_id","quantity"])

print("customers:", customers.shape, "| products:", products.shape,
      "| orders:", orders.shape, "| order_items:", order_items.shape)
orders.head()

## 3 · Series vs DataFrame

A **Series** is a labeled 1-D array (one column). A **DataFrame** is a dict-like
table of aligned Series sharing an index.

In [ ]:
s = orders["amount"]                 # a Series
print(type(s).__name__)
print(s.head(3))
print("describe:\n", s.describe().round(2))

## 4 · DataFrame basics: shape, dtypes, info, describe

First look at any dataset: `head`, `shape`, `dtypes`, `info()`, `describe()`.

In [ ]:
print("shape:", orders.shape)
print("\ndtypes:\n", orders.dtypes)
print("\nnumeric summary:\n", orders.describe().round(2))
customers.head()

## 5 · Selecting columns

One column → a Series (`df["col"]`); several → a DataFrame (`df[["a","b"]]`).

In [ ]:
print(orders["status"].head(3).tolist())
orders[["order_id", "amount", "status"]].head()

## 6 · Filtering rows

Select rows with a **boolean mask**. Combine conditions with `&` / `|` (wrap each
in parentheses). `isin`, `between`, and `.query()` are handy shortcuts.

In [ ]:
big = orders[(orders["amount"] >= 200) & (orders["status"] == "completed")]
print("big completed orders:", len(big))

print(customers[customers["country"].isin(["US", "GB"])][["name","country"]])

print(orders.query("amount > 300 and status == 'completed'").shape)

## 7 · `.loc` vs `.iloc`

- **`.loc[rows, cols]`** selects by **label** (and is end-**inclusive** in slices).
- **`.iloc[rows, cols]`** selects by **integer position** (end-exclusive).

In [ ]:
print(orders.iloc[0])                                  # first row by position
print("---")
print(orders.iloc[0:3, [0, 4]])                        # rows 0-2, cols 0 & 4
print("---")
print(orders.loc[orders["amount"] >= 350, ["order_id","amount","status"]].head())

## 8 · Deriving new columns (vectorized)

Assign a new column from an expression over existing ones — applied to the whole
column at once. `np.where` handles conditionals; `assign` returns a new frame.

In [ ]:
orders["amount_tier"] = np.where(orders["amount"] >= 200, "big", "small")
orders["net"] = (orders["amount"] * 1.2).round(2)
orders["is_completed"] = orders["status"].eq("completed")
orders[["order_id","amount","amount_tier","net","is_completed"]].head()

## 9 · Sorting

`sort_values` orders rows; `nlargest`/`nsmallest` grab the extremes fast.

In [ ]:
print(orders.sort_values("amount", ascending=False)
            [["order_id","amount","status"]].head())
print("--- top 3 by amount ---")
print(orders.nlargest(3, "amount")[["order_id","amount"]])

## 10 · Missing data

Real data has gaps. `isna` flags them, `fillna` fills, `dropna` removes. Decide
per column whether missing means drop, default, or flag. Our `customers.email`
has some `None`.

In [ ]:
print("nulls per column:\n", customers.isna().sum())

customers["has_email"] = customers["email"].notna()
filled = customers.assign(email=customers["email"].fillna("UNKNOWN"))
print("\nafter fillna, nulls:", int(filled["email"].isna().sum()))
print(filled[["name","email","has_email"]])

## 11 · Aggregations & `value_counts`

Series methods give instant summaries; `value_counts` tallies categories.

In [ ]:
print("total revenue:", round(orders.loc[orders["is_completed"], "amount"].sum(), 2))
print("avg order:", round(orders["amount"].mean(), 2))
print("\nstatus counts:")
print(orders["status"].value_counts())
print("\nshare:")
print(orders["status"].value_counts(normalize=True).round(3))

## 12 · `groupby` — split, apply, combine

Group rows by a key, compute an aggregate per group. `agg` runs several at once
and names the outputs (**named aggregation**).

In [ ]:
by_status = (orders
    .groupby("status")
    .agg(n_orders=("order_id", "count"),
         revenue=("amount", "sum"),
         avg_amount=("amount", "mean"))
    .round(2)
    .reset_index())
print(by_status)

## 13 · `apply` and `map`

When a transform isn't a built-in vectorized op: `map` applies a function
element-wise to a **Series**; `apply` applies one along a DataFrame's rows
(`axis=1`) or columns. Prefer vectorized ops when you can — they're faster.

In [ ]:
orders["band"] = orders["amount"].map(
    lambda a: "high" if a >= 250 else "mid" if a >= 100 else "low")
print(orders["band"].value_counts())

orders["tag"] = orders.apply(lambda r: f"{r['status'][:4]}:{r['band']}", axis=1)
print(orders[["amount","status","band","tag"]].head())

## 14 · The `.str` accessor — text columns

`.str` vectorizes Python string methods over a column (NaN-safe): `lower`,
`strip`, `contains`, `split`, `extract`, and more. We clean the messy country
codes and pull the email domain.

In [ ]:
customers["country_clean"] = customers["country"].str.strip().str.upper()
customers["email_domain"] = customers["email"].str.extract(r"@(.+)$")
print(customers[["name","country","country_clean","email","email_domain"]])

## 15 · The `.dt` accessor — datetime columns

`.dt` exposes datetime parts (`year`, `month`, `dayofweek`), and `to_period`
buckets timestamps. `pd.to_datetime` parses strings into real timestamps first.

In [ ]:
orders["order_date"] = orders["order_ts"].dt.date
orders["order_month"] = orders["order_ts"].dt.to_period("M").astype(str)
orders["weekday"] = orders["order_ts"].dt.day_name()
print(orders[["order_ts","order_date","order_month","weekday"]].head())

## 16 · `merge` — SQL-style joins

Combine tables on a key. `how=` picks the join type (`inner`/`left`/`right`/
`outer`), exactly like SQL. First clean the join key, then attach each order's
country and roll up revenue by country.

In [ ]:
cust = customers.assign(country=customers["country"].str.strip().str.upper())

orders_enriched = orders.merge(
    cust[["customer_id","country"]], on="customer_id", how="left")

rev_by_country = (orders_enriched[orders_enriched["is_completed"]]
    .groupby("country")["amount"].sum().round(2)
    .sort_values(ascending=False))
print(rev_by_country)

In [ ]:
# Multi-table join: line items -> products -> category revenue
line_rev = order_items.merge(products, on="product_id", how="left")
line_rev["line_amount"] = line_rev["quantity"] * line_rev["unit_price"]
print(line_rev.groupby("category")["line_amount"].sum().round(2)
              .sort_values(ascending=False))

## 17 · `concat` — stacking frames

`merge` joins side-by-side on a key; `concat` **stacks** rows on top of each
other (same columns) — how you union daily files or append a batch.

In [ ]:
jan = orders[orders["order_ts"].dt.month == 1].head(2)
feb = orders[orders["order_ts"].dt.month == 2].head(2)
stacked = pd.concat([jan, feb], ignore_index=True)
print("rows:", len(jan), "+", len(feb), "->", len(stacked))
print(stacked[["order_id","order_ts","amount"]])

## 18 · Duplicates

`duplicated` flags repeat rows; `drop_duplicates` removes them. Use `subset=` to
define "duplicate" by specific columns and `keep=` to choose which copy to keep.

In [ ]:
demo = pd.DataFrame({
    "customer_id": [1, 1, 2, 2, 2],
    "email": ["a@x","a@x","b@x","b@x","b2@x"],
})
print("duplicated rows:", int(demo.duplicated().sum()))
print(demo.drop_duplicates())
print("--- one row per customer (keep last) ---")
print(demo.drop_duplicates(subset="customer_id", keep="last"))

## 19 · Reshaping: `pivot_table` and `melt`

**Wide** vs **long** format. `pivot_table` spreads a key into columns (reports);
`melt` collapses columns back into rows (tidy storage).

In [ ]:
wide = pd.pivot_table(
    orders_enriched,
    index="country", columns="status", values="amount",
    aggfunc="sum", fill_value=0).round(0)
print("pivot (revenue by country x status):")
print(wide.head())

long = wide.reset_index().melt(id_vars="country", value_name="revenue")
print("\nmelted back to long:")
print(long.head())

## 20 · Time series — `resample`, `rolling`, `shift`

With a **datetime index**, pandas does calendar-aware analytics. `resample` is
`groupby` for time; `rolling` gives moving windows; `shift` compares to a prior
period.

In [ ]:
ts = (orders[orders["is_completed"]]
      .set_index("order_ts").sort_index())

monthly = ts["amount"].resample("MS").sum().round(2)     # MS = month start
print("monthly revenue:\n", monthly.head())

daily = ts["amount"].resample("D").sum()
ma7 = daily.rolling(window=7, min_periods=1).mean().round(2)
print("\n7-day moving average (head):\n", ma7.head(8))

mom = monthly.to_frame("revenue")
mom["prev"] = mom["revenue"].shift(1)
mom["mom_pct"] = ((mom["revenue"] - mom["prev"]) / mom["prev"] * 100).round(1)
print("\nmonth-over-month:\n", mom.head())

## 21 · Categorical dtype

Columns with few distinct values (status, country) stored as `category` use far
less memory and speed up grouping — pandas stores each label once.

In [ ]:
before = orders["status"].memory_usage(deep=True)
orders["status"] = orders["status"].astype("category")
after = orders["status"].memory_usage(deep=True)
print("dtype now:", orders["status"].dtype)
print("categories:", list(orders["status"].cat.categories))
print(f"memory: {before} -> {after} bytes")

## 22 · Reading & writing data (IO)

pandas reads/writes CSV, JSON, Parquet and more. Here we round-trip **CSV**
through an in-memory buffer (no file needed). On Databricks you'd read from a
path or volume, e.g. `pd.read_csv("/Volumes/.../orders.csv")`.

In [ ]:
import io
buf = io.StringIO()
by_status.to_csv(buf, index=False)          # write CSV to a string buffer
print("CSV text:\n" + buf.getvalue())

reloaded = pd.read_csv(io.StringIO(buf.getvalue()))
print("read back:\n", reloaded)

# For big data, prefer Parquet (columnar, typed, compressed):
#   df.to_parquet("data.parquet");  pd.read_parquet("data.parquet")

## 23 · Bonus: pandas on Databricks

pandas runs on the **driver** (one machine) — perfect for data that fits in
memory. When data is too big, use **Spark**; converting between the two is easy:

```python
# pandas DataFrame  ->  Spark DataFrame (distributed)
sdf = spark.createDataFrame(orders)
display(sdf)                     # rich interactive table in Databricks

# Spark DataFrame  ->  pandas (only when it fits on the driver!)
pdf = sdf.limit(1000).toPandas()

# Best of both: the pandas API that runs on Spark (distributed pandas)
import pyspark.pandas as ps
psdf = ps.DataFrame(orders)      # same pandas syntax, Spark execution
psdf.groupby("status")["amount"].sum()
```

**Rule of thumb:** prototype and handle small/medium data with pandas; switch to
Spark (or `pyspark.pandas`) when it no longer fits on one machine. The concepts
in this notebook transfer directly.

## 🎓 You've covered pandas end to end

NumPy vectorization, `Series`/`DataFrame`, selection with `.loc`/`.iloc`,
filtering, deriving columns, sorting, missing data, `groupby`/`agg`, `apply`/
`map`, the `.str` and `.dt` accessors, `merge`/`concat`, duplicates,
`pivot`/`melt`, time series, categoricals, and IO.

**Next:** try these on your own data, and explore `pyspark.pandas` to scale the
exact same code on Databricks. Happy wrangling! 🚀